# Book Market Analytics – Books to Scrape
**Data Science 500 – Project Notebook**

**Student name:** JOSHUA MAHASHA  |  **Student ITS No:** 402605615

**Business question:** Which factors influence book prices and ratings on Books to Scrape, and how well can price be predicted from public book information?

**Pipeline:** 1. Data acquisition → 2. Cleaning → 3. Visualisation and exploration → 4. Linear regression → 5. Interpretation and limitations

> Run the notebook top to bottom. Section 1 scrapes about 1 000 books (roughly 4–6 minutes because of a polite delay between requests) and saves `books_raw.csv`. If that file already exists, scraping is skipped so the notebook is reproducible.

In [ ]:
import re, time, os
import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from bs4 import BeautifulSoup
from urllib.parse import urljoin
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 30)
RANDOM_STATE = 42
BASE_URL = "https://books.toscrape.com/"
RAW_FILE = "books_raw.csv"
CLEAN_FILE = "books_clean.csv"

## 1. Data acquisition (web scraping)

**Approach**
1. Walk through every catalogue listing page (`catalogue/page-1.html` … `page-50.html`) and collect each book's link.
2. Open each book's detail page and extract: **title, price (GBP), star rating (word), stock availability and quantity**, plus extra fields useful for modelling: **category (genre), description, UPC, number of reviews**.
3. Store the values exactly as scraped (raw text) so that all cleaning decisions are visible and justified in Section 2.

The listing page only shows title, price, rating and an in-stock flag. The genre, description and stock quantity live on the detail page, which is why both levels are scraped.

In [ ]:
session = requests.Session()
session.headers.update({"User-Agent": "Mozilla/5.0 (student data science project)"})

def get_soup(url, retries=3, delay=1.0):
    """Download a page and return a BeautifulSoup object (with retries)."""
    for attempt in range(retries):
        try:
            resp = session.get(url, timeout=15)
            resp.encoding = "utf-8"          # avoids the 'Â£' encoding problem in prices
            resp.raise_for_status()
            return BeautifulSoup(resp.text, "html.parser")
        except requests.RequestException as err:
            print(f"  retry {attempt + 1} for {url}: {err}")
            time.sleep(delay * (attempt + 1))
    return None

def parse_listing_page(soup, page_url):
    """Return (title, price, rating, availability, detail_url) for each book on a listing page."""
    rows = []
    for article in soup.select("article.product_pod"):
        link = article.h3.a
        rows.append({
            "title_listing": link["title"],
            "price_raw": article.select_one("p.price_color").get_text(strip=True),
            "rating_raw": article.select_one("p.star-rating")["class"][1],   # e.g. 'Three'
            "availability_listing": article.select_one("p.availability").get_text(strip=True),
            "detail_url": urljoin(page_url, link["href"]),
        })
    return rows

def parse_detail_page(soup):
    """Extract the extra fields from a book detail page."""
    crumbs = soup.select("ul.breadcrumb li a")
    category = crumbs[-1].get_text(strip=True) if len(crumbs) >= 3 else None
    desc_tag = soup.select_one("#product_description")
    description = desc_tag.find_next_sibling("p").get_text(strip=True) if desc_tag else None
    table = {r.th.get_text(strip=True): r.td.get_text(strip=True)
             for r in soup.select("table.table-striped tr")}
    return {
        "category": category,
        "description": description,
        "upc": table.get("UPC"),
        "availability_detail": table.get("Availability"),
        "num_reviews_raw": table.get("Number of reviews"),
    }

In [ ]:
def scrape_catalogue(max_pages=50, pause=0.2):
    records = []
    for page in range(1, max_pages + 1):
        page_url = urljoin(BASE_URL, f"catalogue/page-{page}.html")
        soup = get_soup(page_url)
        if soup is None:
            print(f"Skipping listing page {page}")
            continue
        listing = parse_listing_page(soup, page_url)
        for item in listing:
            detail = get_soup(item["detail_url"])
            item.update(parse_detail_page(detail) if detail else {})
            records.append(item)
            time.sleep(pause)
        print(f"page {page:>2}/{max_pages}: {len(records)} books so far", end="\r")
    return pd.DataFrame(records)

if os.path.exists(RAW_FILE):
    raw = pd.read_csv(RAW_FILE)
    print(f"Loaded existing {RAW_FILE}: {raw.shape}")
else:
    raw = scrape_catalogue()
    raw.to_csv(RAW_FILE, index=False)
    print(f"\nScraped and saved {RAW_FILE}: {raw.shape}")
raw.head()

In [ ]:
print("Rows scraped:", len(raw), "(brief requires at least 200)")
raw.info()

## 2. Data cleaning and preparation

Each decision is stated with its reason.

| Issue in the raw data | Decision | Why |
|---|---|---|
| Price stored as text like `£51.77` | Strip currency symbol, convert to `float` | Needed as the numeric target |
| Rating stored as a word (`One`…`Five`) | Map to integers 1–5 | Ordinal, so a numeric scale is meaningful |
| Availability stored as `In stock (22 available)` | Extract the quantity into `stock_qty` and a boolean `in_stock` | Quantity is a usable numeric feature |
| Some books have no description | Fill with an empty string and flag with `has_description` | Missing here means "not provided"; dropping rows would lose valid price data |
| Whitespace and odd characters in text | Strip and normalise | Consistent text features |
| Duplicates | Drop on UPC | The same book must not be counted twice |
| Numeric text features | Create description length, word count, title length | Simple, explainable text features |

In [ ]:
df = raw.copy()

print("Missing values per column (raw):")
print(df.isna().sum())
print("\nDuplicate UPCs:", df["upc"].duplicated().sum())

In [ ]:
RATING_MAP = {"One": 1, "Two": 2, "Three": 3, "Four": 4, "Five": 5}

# Title: prefer the full title from the listing page's title attribute
df["title"] = df["title_listing"].astype(str).str.strip()

# Price: keep only digits and the decimal point
df["price"] = pd.to_numeric(df["price_raw"].astype(str).str.replace(r"[^0-9.]", "", regex=True), errors="coerce")

# Rating: word -> integer
df["rating"] = df["rating_raw"].map(RATING_MAP)

# Availability: use the detail-page text (it carries the quantity)
avail = df["availability_detail"].fillna(df["availability_listing"]).astype(str)
df["in_stock"] = avail.str.contains("In stock", case=False)
df["stock_qty"] = pd.to_numeric(avail.str.extract(r"(\d+)")[0], errors="coerce").fillna(0).astype(int)

# Reviews
df["num_reviews"] = pd.to_numeric(df["num_reviews_raw"], errors="coerce").fillna(0).astype(int)

# Category
df["category"] = df["category"].fillna("Unknown").str.strip()

# Description: missing -> empty string + flag
df["description"] = df["description"].fillna("").astype(str).str.strip()
df["has_description"] = (df["description"] != "").astype(int)

# Duplicates
before = len(df)
df = df.drop_duplicates(subset="upc").reset_index(drop=True)
print(f"Dropped {before - len(df)} duplicate rows")

# Basic validity checks
df = df.dropna(subset=["price", "rating"])
df = df[df["price"] > 0].reset_index(drop=True)

In [ ]:
# Text-derived features
df["title_length"] = df["title"].str.len()
df["title_words"] = df["title"].str.split().str.len()
df["desc_length"] = df["description"].str.len()
df["desc_words"] = df["description"].str.split().str.len().fillna(0).astype(int)
df["title_has_colon"] = df["title"].str.contains(":").astype(int)

clean = df[["title", "category", "price", "rating", "in_stock", "stock_qty", "num_reviews",
            "has_description", "desc_length", "desc_words", "title_length", "title_words",
            "title_has_colon", "upc", "description"]].copy()

clean.to_csv(CLEAN_FILE, index=False)
print("Clean dataset saved:", CLEAN_FILE, clean.shape)
clean.describe(include="all").T.head(15)

In [ ]:
print("Missing values after cleaning:\n", clean.isna().sum()[clean.isna().sum() > 0])
print("\nnum_reviews unique values:", clean["num_reviews"].unique())
print("in_stock value counts:\n", clean["in_stock"].value_counts())
print("Categories:", clean["category"].nunique())

**Observation to carry forward:** on this sandbox site the number of reviews is constant (0) and almost every book is in stock, so those columns carry little or no predictive information. They are kept in the dataset for completeness but are excluded from the model (justified in Section 4).

## 3. Data visualisation and exploration

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
sns.histplot(clean["price"], bins=30, kde=True, ax=axes[0], color="#2c6fbb")
axes[0].set(title="Distribution of book prices", xlabel="Price (GBP)", ylabel="Number of books")
sns.countplot(x="rating", data=clean, ax=axes[1], color="#2c6fbb")
axes[1].set(title="Number of books per star rating", xlabel="Star rating", ylabel="Number of books")
plt.tight_layout(); plt.show()
print(clean["price"].describe().round(2))

In [ ]:
plt.figure(figsize=(8, 4.5))
sns.boxplot(x="rating", y="price", data=clean, color="#9ec5ee")
plt.title("Price by star rating"); plt.xlabel("Star rating"); plt.ylabel("Price (GBP)")
plt.show()
print(clean.groupby("rating")["price"].agg(["count", "mean", "median"]).round(2))

In [ ]:
cat_stats = (clean.groupby("category")["price"].agg(["count", "mean", "median"]).round(2)
             .sort_values("mean", ascending=False))
top_n = 12
fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))
cat_stats.head(top_n)["mean"].iloc[::-1].plot(kind="barh", ax=axes[0], color="#2c6fbb")
axes[0].set(title=f"Top {top_n} categories by mean price", xlabel="Mean price (GBP)", ylabel="")
cat_stats[cat_stats["count"] >= 10].tail(top_n)["mean"].plot(kind="barh", ax=axes[1], color="#d98a3d")
axes[1].set(title=f"Lowest {top_n} categories by mean price (10+ books)", xlabel="Mean price (GBP)", ylabel="")
plt.tight_layout(); plt.show()

In [ ]:
top_cats = clean["category"].value_counts().head(10).index
plt.figure(figsize=(11, 5))
sns.boxplot(data=clean[clean["category"].isin(top_cats)], x="category", y="price", color="#9ec5ee")
plt.xticks(rotation=40, ha="right")
plt.title("Price spread within the 10 largest categories"); plt.xlabel(""); plt.ylabel("Price (GBP)")
plt.tight_layout(); plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
for ax, col, label in zip(axes, ["stock_qty", "desc_length", "title_length"],
                          ["Stock quantity", "Description length (characters)", "Title length (characters)"]):
    sns.scatterplot(data=clean, x=col, y="price", alpha=0.35, ax=ax, color="#2c6fbb")
    ax.set(title=f"Price vs {label.lower()}", xlabel=label, ylabel="Price (GBP)")
plt.tight_layout(); plt.show()

In [ ]:
num_cols = ["price", "rating", "stock_qty", "desc_length", "desc_words", "title_length", "title_words"]
plt.figure(figsize=(7.5, 6))
sns.heatmap(clean[num_cols].corr(), annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Correlation matrix of numeric variables"); plt.show()

### Interpretation of the visual findings
_Write this after you have run the cells and looked at your own charts. Do not copy numbers from anywhere else. Suggested prompts:_

- **Price distribution:** shape, range, typical price (use the `describe()` output).
- **Rating vs price:** does the median price move with rating, or is the relationship flat?
- **Genre:** which categories are priced highest and lowest, and how much do prices vary inside a category?
- **Other variables:** do stock quantity, description length or title length show any visible pattern with price?
- **Correlations:** which, if any, correlations are noticeably different from zero?

## 4. Predictive modelling – linear regression

**Target:** `price` (GBP)

**Features and justification**
- `rating` – ordinal 1–5, the main quality signal visible to buyers.
- `stock_qty` – may reflect supply or demand differences.
- `desc_length`, `title_length`, `title_has_colon` – simple text features: richer marketing copy or subtitled titles might sit with different price points.
- `category` – one-hot encoded; genre is the most commercially meaningful descriptor. Categories with fewer than 15 books are grouped as `Other` to avoid unstable coefficients.

**Excluded:** `num_reviews` (constant), `in_stock` (almost constant), `desc_words` and `title_words` (near-duplicates of the length features, which would add multicollinearity), `upc` and `title` (identifiers).

In [ ]:
model_df = clean.copy()
counts = model_df["category"].value_counts()
rare = counts[counts < 15].index
model_df["category_grouped"] = model_df["category"].where(~model_df["category"].isin(rare), "Other")

numeric_features = ["rating", "stock_qty", "desc_length", "title_length", "title_has_colon"]
X = pd.concat([model_df[numeric_features],
               pd.get_dummies(model_df["category_grouped"], prefix="cat", drop_first=True, dtype=int)], axis=1)
y = model_df["price"]
print("Modelling matrix:", X.shape)
print("Categories kept:", model_df["category_grouped"].nunique())

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=RANDOM_STATE)

lin_reg = LinearRegression().fit(X_train, y_train)
pred_train = lin_reg.predict(X_train)
pred_test = lin_reg.predict(X_test)

def report(name, y_true, y_pred):
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    print(f"{name:<6} MAE = £{mean_absolute_error(y_true, y_pred):.2f} | RMSE = £{rmse:.2f} | R² = {r2_score(y_true, y_pred):.3f}")

report("Train", y_train, pred_train)
report("Test", y_test, pred_test)

# Baseline: always predict the training mean
baseline = np.full(len(y_test), y_train.mean())
report("Base", y_test, baseline)

cv = cross_val_score(LinearRegression(), X, y, cv=5, scoring="r2")
print(f"\n5-fold CV R²: mean = {cv.mean():.3f}, std = {cv.std():.3f}")

In [ ]:
coefs = pd.Series(lin_reg.coef_, index=X.columns).sort_values()
print(f"Intercept: {lin_reg.intercept_:.2f}\n")
top = pd.concat([coefs.head(8), coefs.tail(8)])
plt.figure(figsize=(8, 6))
top.plot(kind="barh", color=["#d98a3d" if v < 0 else "#2c6fbb" for v in top])
plt.axvline(0, color="black", lw=0.8)
plt.title("Largest linear regression coefficients (GBP per unit)"); plt.tight_layout(); plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.8))
axes[0].scatter(y_test, pred_test, alpha=0.5, color="#2c6fbb")
lims = [min(y_test.min(), pred_test.min()), max(y_test.max(), pred_test.max())]
axes[0].plot(lims, lims, "r--", label="Perfect prediction")
axes[0].set(title="Actual vs predicted price (test set)", xlabel="Actual (GBP)", ylabel="Predicted (GBP)")
axes[0].legend()
residuals = y_test - pred_test
axes[1].scatter(pred_test, residuals, alpha=0.5, color="#2c6fbb")
axes[1].axhline(0, color="red", ls="--")
axes[1].set(title="Residuals vs predicted", xlabel="Predicted (GBP)", ylabel="Residual (GBP)")
plt.tight_layout(); plt.show()

## 5. Interpretation and limitations
_Complete this section from **your own** output. Points to address:_

1. **Performance:** compare test MAE/RMSE/R² with the mean-only baseline and with the training scores. Is the model meaningfully better than guessing the average price?
2. **Generalisation:** do train and test scores agree, and is the cross-validated R² stable across folds?
3. **Coefficients:** which features have the largest effects, in £ per unit? Are they large relative to the typical price spread? Treat them as associations, not causes.
4. **Residual plots:** is the spread of errors roughly constant, or does the model struggle with particular price ranges?
5. **Limitations to discuss:**
   - Books to Scrape is a **sandbox** site with synthetic prices and ratings, so findings do not transfer to real retail pricing.
   - Weak features: no author, publisher, page count, publication year or format are available.
   - Linear regression assumes additive linear effects and is sensitive to the categories grouped as `Other`.
   - Multiple categories have small samples, giving wide uncertainty on their coefficients.
6. **Recommendations** for retailers, publishers and book buyers should follow from what the data actually shows, including a candid statement if price turns out to be largely unpredictable from these features.